# 第 1 周第 2 天 —— 用 GPT 解读 Kubernetes Pod 文档

## 练习目标（理念）

把网页正文抓下来，交给 **Chat Completions API**，让模型用 Markdown 给出简洁的技术说明（这里聚焦 Pod：是什么、怎么跑、给一个例子）。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 环境变量与 `.env` | `load_dotenv` + `OPENAI_API_KEY` |
| `system` / `user` messages | DevOps 角色 + 网页正文 |
| 网页抓取 | `fetch_website_contents`（来自同目录 `scraper`） |
| 调用云端模型 | `gpt-5-nano`（常量 `model_agent`） |

## 怎么跑

1. 确保同目录有可用的 `scraper.py`，且 `.env` 里配置了 `OPENAI_API_KEY`
2. 从上到下依次运行单元格（Shift+Enter）
3. 最后一格会对 Kubernetes Pod 文档页做摘要；也可把 URL 换成其他技术文档试一试


In [14]:
# ========== 导入：后面抓网页、调 OpenAI、在笔记本里展示 Markdown 都要用到 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进进程环境，避免把密钥写进代码
from dotenv import load_dotenv
# 从同目录 scraper 导入抓取函数：把网页正文取出来交给模型
from scraper import fetch_website_contents
# 从 IPython.display 导入展示工具：在笔记本里把模型回复渲染成 Markdown
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI


In [15]:
# ========== 加载 .env 并检查 API Key 格式 ==========

# 加载环境变量：override=True 表示用 .env 覆盖已有同名环境变量
load_dotenv(override=True)
# 从环境变量读取 OpenAI API Key（名字必须是 OPENAI_API_KEY，和官方 SDK 默认一致）
api_key = os.getenv('OPENAI_API_KEY')
# 后面 create() 会用的模型 id：集中写成常量，方便改
model_agent = "gpt-5-nano"
# ========== 检查钥匙：常见配置错误提前报出来 ==========

# 情况 1：根本没读到 Key
if not api_key:
    # 错误提示字符串保持英文原样：程序/课程排错文案，勿改译
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
# 情况 2：Key 不像项目密钥（常见前缀 sk-proj-）
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
# 情况 3：首尾有空格/制表符，容易导致鉴权失败
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    # 基本检查通过（仍不保证 Key 一定有效，只是格式看起来合理）
    print("API key found and looks good so far!")


API key found and looks good so far!


In [16]:
# ========== Prompt：system 定角色/风格，user 定具体任务 ==========

# system_prompt：发给模型的「角色与写作规范」；字符串内容勿改译（改了会改变回答风格）
system_prompt= """You are a DevOps expert. You write with precision, clarity, and conciseness.
analyzes the contents of a website, and provides a short clear thecnical Respond in markdown, 
Do not wrap the markdown in a code block - respond just with the markdown, Use examples when helpful
Avoid unnecessary explanations.
"""

# user_prompt：用户侧任务说明；后面会再拼接网页正文
user_prompt=""" 
Here are the contents of a website.
Provide a short summary of what pods are and how to run it. Provide only one example.
"""


In [ ]:
# ========== 客户端 + 三个函数：组 messages → 调 API → 展示 ==========

# 创建 OpenAI 客户端：默认从环境变量 OPENAI_API_KEY 取密钥
openai = OpenAI()

# 组装 Chat Completions 所需的 messages 列表（system + user）
def create_prompt(website):
    return [
        # system：角色与输出格式约束
        {"role": "system", "content": system_prompt},
        # user：固定任务说明 + 抓到的网页正文
        {"role": "user", "content": user_prompt + website}
    ]  

# 抓取 URL 对应网页，调用模型，返回助手回复文本
def summarize(url):
    # 用 scraper 拉取网页正文（字符串）
    website = fetch_website_contents(url)
    # 调用 Chat Completions：model / messages 决定「用谁答、答什么」
    response = openai.chat.completions.create(
        model = model_agent,
        messages = create_prompt(website)
    )
    # choices[0].message.content：第一条候选回复的正文
    return response.choices[0].message.content

# 把摘要渲染成 Markdown 显示在笔记本里
def display_summary(url):
    # 先拿到模型生成的 Markdown 字符串
    summary = summarize(url)
    # display(Markdown(...))：在 Jupyter 中漂亮展示
    display(Markdown(summary))


In [20]:
# ========== 实跑：对 Kubernetes Pod 概念页做技术摘要 ==========

# URL 保持原样；也可换成其他文档页做对比
display_summary("https://kubernetes.io/docs/concepts/workloads/pods")


- Pods are the smallest deployable units in Kubernetes. A Pod runs one or more containers in a shared context (same network namespace and storage). Each Pod gets its own IP and port space, and Pods are scheduled onto nodes. They’re lightweight and designed to be ephemeral, usually managed by higher-level controllers (e.g., Deployments).

- How to run a Pod:
  - Create a Pod manifest (YAML) and apply it with kubectl.
  - Verify status and view logs as needed.

- Example: single-container Pod running nginx
  apiVersion: v1
  kind: Pod
  metadata:
    name: nginx-pod
  spec:
    containers:
    - name: nginx
      image: nginx:stable-alpine
      ports:
      - containerPort: 80

  Commands:
  - kubectl apply -f pod.yaml
  - kubectl get pods
  - kubectl describe pod nginx-pod
  - kubectl logs nginx-pod